In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.set_theme(style="whitegrid", font="cmr10", font_scale=1.9)
plt.rcParams["axes.formatter.use_mathtext"] = True
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})
colors = sns.color_palette("dark")
colors

In [ ]:
import os
def load_from_csv(path):
    output = []
    if not os.path.exists(path):
        return np.array(output)
    with open(path, 'r') as f:
        for line in f:
            output.append(float(line.strip()))
    return np.array(output)

def load_from_csv_split(path):
    output = []
    if not os.path.exists(path):
        return np.array(output)
    with open(path, 'r') as f:
        for line in f:
            split = line.strip().split(',')
            if len(output) == 0:
                for i in range(len(split)):
                    output.append([])
            for i in range(len(split)):
                output[i].append(float(split[i].strip()))
    for i in range(len(output)):
        output[i] = np.array(output[i])
    return output

In [ ]:
ratio = 0.05
lambda_values = [0.03, 0.05, 0.1, 0.2]
seeds = [1, 2, 3]

data_by_lambda = {lam: {"loss": [], "acc": [], "step_list": None} for lam in lambda_values}

for seed in seeds:
    for lam in lambda_values:
        directory = f"datas/{seed}/ratio_{ratio}_lambda_{lam}_lr_5.0e-06"
        loss_array = load_from_csv(f"{directory}/loss.csv")
        acc_array = load_from_csv(f"{directory}/accuracy.csv")
        steps_array = load_from_csv(f"{directory}/steps.csv")
        if len(acc_array) > 0:
            data_by_lambda[lam]["loss"].append(loss_array)
            data_by_lambda[lam]["acc"].append(acc_array)
            if data_by_lambda[lam]["step_list"] is None:
                data_by_lambda[lam]["step_list"] = steps_array
            print(f"  ✓ seed={seed}, λ={lam}")
        else:
            print(f"  ✗ seed={seed}, λ={lam}")

In [ ]:
background_predictability = 70

fig, axs = plt.subplots(1, 1, figsize=(13, 5.3))

ran = 1800
for idx, lam in enumerate(lambda_values):
    d = data_by_lambda[lam]
    if len(d["acc"]) == 0:
        continue

    min_len = min(len(a) for a in d["acc"])
    acc_stack = np.stack([a[:min_len] for a in d["acc"]])  # (n_seeds, steps)
    steps = d["step_list"][:min_len]

    acc_mean = acc_stack[:, :ran].mean(axis=0)
    acc_std = acc_stack[:, :ran].std(axis=0)
    steps_plot = steps[:ran]

    stride = 2
    axs.plot(steps_plot, acc_mean,
             alpha=0.7, linewidth=3, color=colors[idx],
             label=f"$\\alpha$ = {lam}")
    axs.errorbar(steps_plot[::stride], acc_mean[::stride], yerr=acc_std[::stride],
                 fmt='none', ecolor=colors[idx],
                 elinewidth=2, capsize=3, capthick=1.5, alpha=0.35)

    pos = np.where(acc_mean >= background_predictability * 1.005)
    if len(pos[0]) > 0:
        axs.axvline(steps_plot[pos[0][0]], linestyle="--", color=colors[idx], linewidth=4, alpha=0.7)

axs.text(600, 68.5, "background-label correlation")
axs.set_ylim(65, 77.5)
axs.axhline(background_predictability, linestyle="--", color="black", linewidth=2, alpha=0.6)
axs.set_xlabel("Step")
axs.set_ylabel("Linear Probing Accuracy")
axs.legend()

plt.tight_layout()
plt.savefig("figure/cmnist_mean_std.png", bbox_inches='tight')
plt.show()